In [1]:
import numpy as np
from scipy.stats import norm

In [2]:
S0 = 100
K = 110
H = 90 # Barrier
r = 0.05
sigma = 0.2
T = 1

In [3]:
num_steps = 2_500
num_paths = 50_000
dt = T/num_steps

In [4]:
def bs_prices(S0, r, sigma, dt, num_paths, num_steps):

    z = np.random.randn(num_paths, num_steps)
    log_returns = (r - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z
    log_S = np.log(S0) + np.cumsum(log_returns, axis=-1)

    return np.exp(log_S)

In [5]:
prices = bs_prices(S0, r, sigma, dt, num_paths, num_steps)
np.mean(np.maximum(prices[:, -1]-K, 0))*np.exp(-r*T)

np.float64(6.02095420781464)

In [6]:
def bs_call(K, T, S0, r, sigma):

    d1 = (np.log(S0/K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

In [7]:
bs_call(K, T, S0, r, sigma)

np.float64(6.040088129724239)

In [8]:
num_steps = 2_500
num_paths = 50_000
dt = T/num_steps
prices = bs_prices(S0, r, sigma, dt, num_paths, num_steps)
not_hit_barrier = np.all(prices>H, axis=-1)
np.mean(np.maximum(prices[:, -1]-K, 0)*not_hit_barrier)*np.exp(-r*T)

np.float64(5.323698981532624)

In [9]:
def bs_down_and_out_call(K, H, T, r, sigma):

    alpha = 0.5 * (1- (r/(0.5*sigma**2)))
    term1 = bs_call(K, T, S0, r, sigma)
    term2 = (S0/H)**(2*alpha) * bs_call(K, T, H**2/S0, r, sigma)
    
    return term1 - term2

In [10]:
bs_down_and_out_call(K, H, T, r, sigma)

np.float64(5.296180818795614)

In [11]:
num_steps = 2_500
dt = T/num_steps
num_paths = 50_000
prices = bs_prices(S0, r, sigma, dt, num_paths, num_steps)
np.mean(prices[:, -1]>K)

np.float64(0.36918)

In [12]:
def bs_binary_cash_or_nothing(K, T, r, sigma):

    d1 = (np.log(S0/K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    return np.exp(-r * T) * norm.cdf(d2)

In [13]:
bs_binary_cash_or_nothing(K, T, r, sigma)

np.float64(0.35386095394539413)

In [14]:
%load_ext autoreload
%autoreload 2
from simulators import BlackScholesSimulator

In [15]:
S0 = np.array([50.0, 100.0, 200.0])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
maturity = 1.0
num_steps = 250
num_paths = 5
seed = 42
random_generator = np.random.default_rng(0)
bs = BlackScholesSimulator(S0, r, sigma, maturity, num_steps, num_paths, random_generator)

In [16]:
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
bs.euro_call(K)[:,1,1,0]

array([6.04008813, 6.04008813, 6.04008813, 6.04008813, 6.04008813])

In [17]:
K = np.array([[52.5, 55.0], [105.0, 110.0], [210.0, 220.0]])
H = np.array([[40.0, 45.0], [80.0, 90.0], [160.0, 180.0]])
bs.down_out_call(K, H)[:,1,1,0]

array([5.29618082, 5.29618082, 5.29618082, 5.29618082, 5.29618082])

In [18]:
K = np.array([[52.5, 55.0], [105.0, 110.0], [210.0, 220.0]])
P = np.array([[1.0, 1.0], [1.0, 1.0], [1.0, 1.0]])
bs.cash_or_nothing_call(K, P)[:,1,1,0]

array([0.35386095, 0.35386095, 0.35386095, 0.35386095, 0.35386095])